In [ ]:
import os
import librosa
import soundfile as sf
import numpy as np

DIR_1 = r"C:/Users/admin/Documents/PythonFile/Project/Voice-Activity-Detect/data/processed/speech/non-mixed"
DIR_2  = r"C:/Users/admin/Documents/PythonFile/Project/Voice-Activity-Detect/data/processed/non-speech/non-mixed/music"
DIR_OUT    = r"C:/Users/admin/Documents/PythonFile/Project/Voice-Activity-Detect/data/processed/speech/mixed"


os.makedirs(DIR_OUT, exist_ok=True)

count = 1
for f in os.listdir(DIR_OUT):
    if f.lower().endswith(".wav"):
        try:
            idx = int(os.path.splitext(f)[0])
            count = max(count, idx + 1)
        except:
            pass

print("Start output index:", count)

file_1 = sorted(os.listdir(DIR_1), key=lambda x: int(x.split('.')[0]))
file_2  = sorted(os.listdir(DIR_2),  key=lambda x: int(x.split('.')[0]))

for sp, mu in zip(file_1, file_2):

    path_1 = os.path.join(DIR_1, sp)
    path_2 = os.path.join(DIR_2, mu)

    print("Mixing:", sp, "+", mu)

    audio_1, sr = librosa.load(path_1, sr=None)
    audio_2, sr_mu = librosa.load(path_2, sr=None)

    if sr_mu != sr:
        audio_2 = librosa.resample(audio_2, orig_sr=sr_mu, target_sr=sr)

    min_len = min(len(audio_1), len(audio_2))
    audio_1 = audio_1[:min_len]
    audio_2 = audio_2[:min_len]

    alpha = 1  # audio 1
    beta  = 0.5  # audio 2

    mixed = alpha * audio_1 + beta * audio_2 

    # Normalize tránh vỡ tiếng
    mixed = mixed / np.max(np.abs(mixed) + 1e-8)

    out_path = os.path.join(DIR_OUT, f"{count}.wav")
    sf.write(out_path, mixed, sr)

    print("Saved:", out_path)
    count += 1
